In [3]:
# Local setup script
# !pip install mindspore

import os
from pathlib import Path

def resolve_work_dir():
    candidates = [Path.cwd(), Path.cwd() / "dkt"]
    for candidate in candidates:
        if (candidate / "assistments_2009_2010.csv").exists():
            return candidate
    return Path.cwd()

work_dir = resolve_work_dir()

print(f"Working directory for dataset/model files: {work_dir.resolve()}")
print("Files in directory:")
for file in sorted(os.listdir(work_dir)):
    print(f"  - {file}")

file_path = work_dir / "assistments_2009_2010.csv"
print(f"Dataset path: {file_path}")

# Now run the main DKT script below

Working directory for dataset/model files: /workspace/workspace/dkt
Files in directory:
  - .ipynb_checkpoints
  - assistments_2009_2010.csv
  - dkt_model.ckpt
  - dkt_model.mindir
  - maths_model.ipynb
Dataset path: /workspace/workspace/dkt/assistments_2009_2010.csv


In [4]:
import numpy as np
import pandas as pd
import mindspore as ms
import mindspore.nn as nn
import mindspore.ops as ops
from mindspore import Tensor
from mindspore.dataset import GeneratorDataset
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# Set context
ms.set_context(device_target="GPU")

def resolve_work_dir():
    candidates = [Path.cwd(), Path.cwd() / "dkt"]
    for candidate in candidates:
        if (candidate / "assistments_2009_2010.csv").exists():
            return candidate
    return Path.cwd()

WORK_DIR = resolve_work_dir()

# Load and preprocess data
def load_and_preprocess_data(file_path):
    """
    Load ASSISTments dataset and prepare for DKT
    """
    # Load the dataset
    print("Loading dataset...")
    df = pd.read_csv(file_path)

    print(f"Dataset shape: {df.shape}")

    # Use list_skill_ids as skill column
    skill_column = 'list_skill_ids'

    # Handle missing values and extract first skill
    df[skill_column] = df[skill_column].fillna(df['problem_id'].astype(str))
    df[skill_column] = df[skill_column].astype(str)
    df['skill'] = df[skill_column].apply(lambda x: str(x).split(',')[0] if ',' in str(x) else str(x))

    # Convert correct to binary (0 or 1)
    df['correct'] = df['correct'].fillna(0).astype(int)

    # Filter out invalid skills
    df = df[df['skill'] != 'nan']
    df = df[df['skill'] != '']

    # Count skill frequencies and keep top skills
    print("\nCounting skill frequencies...")
    skill_counts = df['skill'].value_counts()
    top_skills = skill_counts.head(500).index.tolist()  # Use top 500 skills for stability

    print(f"Using top {len(top_skills)} skills (out of {len(skill_counts)} total)")

    # Create skill mapping for top skills
    skill_to_idx = {skill: idx + 1 for idx, skill in enumerate(top_skills)}  # 0 for padding/unknown

    # Filter dataset to only include top skills
    df = df[df['skill'].isin(top_skills)]

    # Sort by user_id and order_id
    df = df.sort_values(['user_id', 'order_id'])

    # Prepare sequences
    student_sequences = []
    student_corrects = []  # Store separately for debugging

    for user_id, group in df.groupby('user_id'):
        # Convert skills to indices
        skills = []
        corrects = []

        for _, row in group.iterrows():
            skill = row['skill']
            correct = row['correct']

            if skill in skill_to_idx:
                skills.append(skill_to_idx[skill])
                corrects.append(correct)

        if len(skills) < 2:  # Need at least 2 interactions
            continue

        # Create interaction encoding: (skill_idx - 1) * 2 + correct + 1
        # We subtract 1 from skill_idx to make it 0-based, then add 1 for padding
        interactions = []
        for skill_idx, correct in zip(skills, corrects):
            # skill_idx is 1-based, convert to 0-based
            skill_idx_0based = skill_idx - 1
            # Create token: skill * 2 + correct + 1 (to avoid 0 which is padding)
            token = skill_idx_0based * 2 + correct + 1
            interactions.append(token)

        student_sequences.append(interactions)
        student_corrects.append(corrects)

    num_skills = len(top_skills)

    print(f"\nNumber of unique skills used: {num_skills}")
    print(f"Number of student sequences: {len(student_sequences)}")
    print(f"Average sequence length: {np.mean([len(s) for s in student_sequences]):.2f}")

    # Debug: Check token ranges
    all_tokens = []
    for seq in student_sequences:
        all_tokens.extend(seq)

    min_token = min(all_tokens)
    max_token = max(all_tokens)
    vocab_size_needed = max_token + 1  # +1 for 0-based

    print(f"Token range: {min_token} to {max_token}")
    print(f"Vocabulary size needed: {vocab_size_needed}")
    print(f"Expected vocab size (num_skills * 2 + 1): {num_skills * 2 + 1}")

    return student_sequences, num_skills, skill_to_idx

def create_train_test_split(sequences, test_ratio=0.2, seed=42):
    """
    Split data into training and testing sets
    """
    np.random.seed(seed)
    n_sequences = len(sequences)
    indices = np.arange(n_sequences)
    np.random.shuffle(indices)

    split_idx = int(n_sequences * (1 - test_ratio))
    train_indices = indices[:split_idx]
    test_indices = indices[split_idx:]

    train_sequences = [sequences[i] for i in train_indices]
    test_sequences = [sequences[i] for i in test_indices]

    print(f"Training sequences: {len(train_sequences)}")
    print(f"Testing sequences: {len(test_sequences)}")

    return train_sequences, test_sequences

def prepare_sequences_for_training(sequences, max_length=50):
    """
    Prepare sequences with padding and create input-target pairs
    """
    processed_sequences = []

    for seq in sequences:
        if len(seq) < 2:
            continue

        # For sequences longer than max_length, create multiple windows
        for start in range(0, len(seq) - 1):
            end = min(start + max_length, len(seq))
            window = seq[start:end]

            if len(window) < 2:
                continue

            # Input: all but last token
            input_seq = window[:-1]
            # Target: skill from last token (extract skill)
            last_token = window[-1]

            # Extract skill index: (token - 1) // 2
            target_skill = (last_token - 1) // 2

            # Pad input sequence
            padded_input = np.zeros(max_length - 1, dtype=np.int32)
            if len(input_seq) > 0:
                padded_input[-len(input_seq):] = input_seq

            processed_sequences.append((padded_input, target_skill))

    return processed_sequences

class DKTDataset:
    """Custom dataset for DKT"""
    def __init__(self, sequences):
        self.sequences = sequences

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        input_seq, target_skill = self.sequences[idx]
        return input_seq, target_skill

class DKTModel(nn.Cell):
    """Deep Knowledge Tracing Model with GRU"""
    def __init__(self, num_skills, hidden_size=64, embedding_dim=32):
        super(DKTModel, self).__init__()

        # Vocabulary size: num_skills * 2 + 1 (for padding/unknown)
        self.vocab_size = num_skills * 2 + 1
        self.num_skills = num_skills

        # Embedding layer
        self.embedding = nn.Embedding(self.vocab_size, embedding_dim, padding_idx=0)

        # GRU layer
        self.gru = nn.GRU(
            embedding_dim,
            hidden_size,
            num_layers=1,
            batch_first=True,
            dropout=0.0
        )

        # Output layer
        self.fc = nn.Dense(hidden_size, num_skills)

        # Activation
        self.sigmoid = nn.Sigmoid()

        print(f"\nModel configuration:")
        print(f"  Vocabulary size: {self.vocab_size}")
        print(f"  Number of skills: {self.num_skills}")
        print(f"  Embedding dim: {embedding_dim}")
        print(f"  Hidden size: {hidden_size}")

    def construct(self, x):
        # x shape: (batch_size, seq_length)
        batch_size, seq_length = x.shape

        # Debug: Check input range
        x_min = ops.ReduceMin()(x)
        x_max = ops.ReduceMax()(x)

        # Embedding
        embedded = self.embedding(x)  # (batch_size, seq_length, embedding_dim)

        # GRU
        gru_output, _ = self.gru(embedded)  # (batch_size, seq_length, hidden_size)

        # Take the last output
        last_output = gru_output[:, -1, :]  # (batch_size, hidden_size)

        # Fully connected layer
        output = self.fc(last_output)  # (batch_size, num_skills)

        # Sigmoid activation for probability
        probabilities = self.sigmoid(output)  # (batch_size, num_skills)

        return probabilities

class DKTWithLoss(nn.Cell):
    """Combine model with loss function"""
    def __init__(self, model, num_skills):
        super(DKTWithLoss, self).__init__()
        self.model = model
        self.num_skills = num_skills
        self.loss_fn = nn.BCELoss(reduction='mean')
        self.one_hot = ops.OneHot()

    def construct(self, x, target_skill):
        # Get predictions
        predictions = self.model(x)  # (batch_size, num_skills)

        # Create one-hot target
        target_one_hot = self.one_hot(target_skill, self.num_skills,
                                      Tensor(1.0, ms.float32), Tensor(0.0, ms.float32))

        # Calculate loss
        loss = self.loss_fn(predictions, target_one_hot)

        return loss

def train_dkt_model(train_sequences, num_skills, epochs=3, batch_size=32, learning_rate=0.001):
    """Train the DKT model"""

    # Prepare training data
    print("\nPreparing training data...")
    train_data = prepare_sequences_for_training(train_sequences, max_length=50)
    print(f"Training samples: {len(train_data)}")

    # Limit training samples for stability
    if len(train_data) > 50000:
        train_data = train_data[:50000]
        print(f"Using first 50,000 samples for stability")

    # Check token ranges in training data
    all_tokens = []
    for seq, _ in train_data:
        all_tokens.extend(seq.tolist())

    min_token = min(all_tokens)
    max_token = max(all_tokens)
    print(f"Training token range: {min_token} to {max_token}")
    print(f"Expected max: {num_skills * 2}")

    # Clip tokens to valid range
    for i, (seq, target) in enumerate(train_data):
        seq_clipped = np.clip(seq, 0, num_skills * 2)
        train_data[i] = (seq_clipped, target)

    # Create dataset
    dataset = DKTDataset(train_data)
    dataloader = GeneratorDataset(
        dataset,
        column_names=['input', 'target'],
        shuffle=True
    ).batch(batch_size, drop_remainder=True)

    # Initialize model
    print("\nInitializing DKT model...")
    model = DKTModel(num_skills, hidden_size=64, embedding_dim=32)

    # Debug: Test model with sample input
    print("\nTesting model with sample input...")
    sample_input = Tensor(np.random.randint(0, num_skills * 2 + 1, size=(2, 49)), ms.int32)
    try:
        sample_output = model(sample_input)
        print(f"Sample output shape: {sample_output.shape}")
        print("✓ Model forward pass successful")
    except Exception as e:
        print(f"✗ Model forward pass failed: {e}")
        return None

    # Loss function
    loss_net = DKTWithLoss(model, num_skills)

    # Optimizer - use simple SGD for stability
    optimizer = nn.SGD(model.trainable_params(), learning_rate=learning_rate)

    # Training step (version-compatible)
    train_network = nn.TrainOneStepCell(loss_net, optimizer)
    train_network.set_train()

    def train_step(x, target):
        return train_network(x, target)

    # Training loop
    print("\nStarting training...")
    losses = []

    for epoch in range(epochs):
        epoch_loss = 0
        step_count = 0

        for batch in dataloader.create_tuple_iterator():
            inputs, targets = batch
            inputs = Tensor(inputs, ms.int32)
            targets = Tensor(targets, ms.int32)

            # Clip inputs to valid range
            inputs = ops.clip_by_value(inputs, Tensor(0, ms.int32), Tensor(num_skills * 2, ms.int32))

            try:
                loss = train_step(inputs, targets)
                epoch_loss += loss.asnumpy()
                step_count += 1
                losses.append(loss.asnumpy())

                if step_count % 50 == 0:
                    print(f"  Step {step_count}, Loss: {loss.asnumpy():.4f}")
            except Exception as e:
                print(f"  Error at step {step_count}: {e}")
                continue

        if step_count > 0:
            avg_loss = epoch_loss / step_count
            print(f"Epoch {epoch + 1}/{epochs}, Average Loss: {avg_loss:.4f}")

    print(f"\nTraining completed with {len(losses)} successful steps")

    return model

def evaluate_model(model, test_sequences, num_skills, skill_to_idx, batch_size=32):
    """Evaluate the trained model"""
    print("\nEvaluating model...")

    if model is None:
        print("Model is None, skipping evaluation")
        return 0.0, [], []

    # Prepare test data
    test_data = prepare_sequences_for_training(test_sequences, max_length=50)
    print(f"Test samples: {len(test_data)}")

    # Limit test samples
    if len(test_data) > 10000:
        test_data = test_data[:10000]

    # Clip tokens
    for i, (seq, target) in enumerate(test_data):
        seq_clipped = np.clip(seq, 0, num_skills * 2)
        test_data[i] = (seq_clipped, target)

    # Create test dataset
    test_dataset = DKTDataset(test_data)
    test_dataloader = GeneratorDataset(
        test_dataset,
        column_names=['input', 'target'],
        shuffle=False
    ).batch(batch_size, drop_remainder=True)

    # Evaluation
    model.set_train(False)
    predictions = []
    targets = []

    for batch in test_dataloader.create_tuple_iterator():
        inputs, batch_targets = batch
        inputs = Tensor(inputs, ms.int32)
        inputs = ops.clip_by_value(inputs, Tensor(0, ms.int32), Tensor(num_skills * 2, ms.int32))

        try:
            batch_predictions = model(inputs)
            predictions.extend(batch_predictions.asnumpy())
            targets.extend(batch_targets.asnumpy())
        except Exception as e:
            print(f"Evaluation error: {e}")
            continue

    if len(predictions) == 0:
        print("No predictions made!")
        return 0.0, predictions, targets

    # Calculate accuracy
    predictions = np.array(predictions)
    targets = np.array(targets)

    predicted_skills = np.argmax(predictions, axis=1)
    accuracy = np.mean(predicted_skills == targets)
    print(f"Test Accuracy: {accuracy:.4f}")

    return accuracy, predictions, targets

def predict_mastery(model, student_interactions, skill_to_idx, num_skills):
    """Predict mastery for a specific student"""
    if model is None:
        return {}

    model.set_train(False)

    # Prepare input sequence
    max_length = 49  # input length
    input_seq = np.zeros(max_length, dtype=np.int32)

    # Convert interactions to tokens
    interactions = []
    for skill, correct in student_interactions:
        skill_idx = skill_to_idx.get(str(skill), 0)
        if skill_idx > 0:  # Valid skill
            skill_idx_0based = skill_idx - 1
            token = skill_idx_0based * 2 + correct + 1
            interactions.append(token)

    # Pad sequence
    if len(interactions) > max_length:
        interactions = interactions[-max_length:]

    if len(interactions) == 0:
        return {}

    input_seq[-len(interactions):] = interactions

    # Clip to valid range
    input_seq = np.clip(input_seq, 0, num_skills * 2)

    # Predict
    input_tensor = Tensor(input_seq.reshape(1, -1), ms.int32)
    mastery_probs = model(input_tensor).asnumpy().flatten()

    # Create mastery dictionary
    idx_to_skill = {idx: skill for skill, idx in skill_to_idx.items()}
    mastery_dict = {}

    for skill_idx in range(num_skills):
        if skill_idx + 1 in idx_to_skill:
            skill = idx_to_skill[skill_idx + 1]
            mastery_dict[skill] = float(mastery_probs[skill_idx])

    return mastery_dict

def save_model(model, num_skills):
    """Save the trained model"""
    if model is None:
        print("Cannot save None model")
        return

    print("\nSaving model...")

    # Create a dummy input for export
    dummy_input = Tensor(np.random.randint(0, num_skills * 2 + 1, size=(1, 49)), ms.int32)

    model_base_path = WORK_DIR / "dkt_model"
    checkpoint_path = WORK_DIR / "dkt_model.ckpt"

    try:
        # Export model
        ms.export(
            model,
            dummy_input,
            file_name=str(model_base_path),
            file_format="MINDIR"
        )
        print(f"Model saved as '{model_base_path}.mindir'")
    except Exception as e:
        print(f"Could not export to MINDIR: {e}")

    # Save checkpoint
    try:
        ms.save_checkpoint(model, str(checkpoint_path))
        print(f"Checkpoint saved as '{checkpoint_path}'")
    except Exception as e:
        print(f"Could not save checkpoint: {e}")

# Main execution
if __name__ == "__main__":
    print("=" * 60)
    print("DEEP KNOWLEDGE TRACING MODEL WITH MINDSPORE")
    print("=" * 60)

    # File path
    file_path = WORK_DIR / "assistments_2009_2010.csv"

    try:
        # Step 1: Load and preprocess data
        sequences, num_skills, skill_to_idx = load_and_preprocess_data(file_path)

        if len(sequences) == 0:
            print("No valid sequences found!")
            exit()

        # Step 2: Split data
        train_sequences, test_sequences = create_train_test_split(sequences, test_ratio=0.2)

        # Step 3: Train model
        trained_model = train_dkt_model(
            train_sequences,
            num_skills,
            epochs=3,
            batch_size=32,
            learning_rate=0.001
        )

        # Step 4: Evaluate model
        if trained_model is not None:
            accuracy, predictions, targets = evaluate_model(
                trained_model,
                test_sequences,
                num_skills,
                skill_to_idx
            )

            # Step 5: Example prediction
            print("\n" + "=" * 60)
            print("EXAMPLE PREDICTION")
            print("=" * 60)

            if skill_to_idx:
                # Get first few skills
                sample_skills = list(skill_to_idx.keys())[:3]
                example_interactions = []

                for i, skill in enumerate(sample_skills):
                    correct = 1 if i % 2 == 0 else 0
                    example_interactions.append((skill, correct))

                mastery = predict_mastery(trained_model, example_interactions, skill_to_idx, num_skills)

                print("\nPredicted mastery for example student:")
                for skill, prob in mastery.items():
                    print(f"  Skill {skill}: {prob:.3f}")

        # Step 6: Save the model
        print("\n" + "=" * 60)
        print("MODEL SAVING")
        print("=" * 60)

        save_model(trained_model, num_skills)

        print("\n" + "=" * 60)
        print("PROCESS COMPLETE!")
        print("=" * 60)

    except FileNotFoundError:
        print(f"\n❌ Error: File '{file_path}' not found!")
        print("Please update the 'file_path' variable.")

    except Exception as e:
        print(f"\n❌ Error occurred: {str(e)}")
        import traceback
        traceback.print_exc()

DEEP KNOWLEDGE TRACING MODEL WITH MINDSPORE
Loading dataset...
Dataset shape: (1011079, 20)

Counting skill frequencies...
Using top 500 skills (out of 22779 total)

Number of unique skills used: 500
Number of student sequences: 7317
Average sequence length: 73.73
Token range: 1 to 1000
Vocabulary size needed: 1001
Expected vocab size (num_skills * 2 + 1): 1001
Training sequences: 5853
Testing sequences: 1464

Preparing training data...
Training samples: 423285
Using first 50,000 samples for stability
Training token range: 0 to 1000
Expected max: 1000

Initializing DKT model...

Model configuration:
  Vocabulary size: 1001
  Number of skills: 500
  Embedding dim: 32
  Hidden size: 64

Testing model with sample input...
Sample output shape: (2, 500)
✓ Model forward pass successful

Starting training...


[ERROR] CORE(381,7605dd677680,python3.7):2026-02-13-10:49:10.928.635 [mindspore/core/utils/file_utils.cc:253] GetRealPath] Get realpath failed, path[/tmp/ipykernel_381/3537678963.py]
[ERROR] CORE(381,7605dd677680,python3.7):2026-02-13-10:49:10.929.317 [mindspore/core/utils/file_utils.cc:253] GetRealPath] Get realpath failed, path[/tmp/ipykernel_381/3537678963.py]


  Step 50, Loss: 0.6930
  Step 100, Loss: 0.6930
  Step 150, Loss: 0.6929
  Step 200, Loss: 0.6929
  Step 250, Loss: 0.6929
  Step 300, Loss: 0.6928
  Step 350, Loss: 0.6928
  Step 400, Loss: 0.6927
  Step 450, Loss: 0.6927
  Step 500, Loss: 0.6927
  Step 550, Loss: 0.6926
  Step 600, Loss: 0.6926
  Step 650, Loss: 0.6926
  Step 700, Loss: 0.6925
  Step 750, Loss: 0.6925
  Step 800, Loss: 0.6924
  Step 850, Loss: 0.6924
  Step 900, Loss: 0.6924
  Step 950, Loss: 0.6923
  Step 1000, Loss: 0.6923
  Step 1050, Loss: 0.6923
  Step 1100, Loss: 0.6922
  Step 1150, Loss: 0.6922
  Step 1200, Loss: 0.6921
  Step 1250, Loss: 0.6921
  Step 1300, Loss: 0.6921
  Step 1350, Loss: 0.6920
  Step 1400, Loss: 0.6920
  Step 1450, Loss: 0.6919
  Step 1500, Loss: 0.6919
  Step 1550, Loss: 0.6919
Epoch 1/3, Average Loss: 0.6925
  Step 50, Loss: 0.6918
  Step 100, Loss: 0.6918
  Step 150, Loss: 0.6917
  Step 200, Loss: 0.6917
  Step 250, Loss: 0.6917
  Step 300, Loss: 0.6916
  Step 350, Loss: 0.6916
  Step 4

In [5]:
# Single-instance inference sanity check
if 'trained_model' not in globals() or trained_model is None:
    print('Run the partial training cell first to create trained_model.')
elif 'test_sequences' not in globals() or len(test_sequences) == 0:
    print('test_sequences not found. Re-run data split in the partial training cell.')
else:
    sample_pairs = prepare_sequences_for_training(test_sequences[:1], max_length=50)
    if len(sample_pairs) == 0:
        print('No valid sample available for single-instance inference.')
    else:
        input_seq, target_skill = sample_pairs[0]
        input_seq = np.clip(input_seq, 0, num_skills * 2)
        input_tensor = Tensor(input_seq.reshape(1, -1), ms.int32)

        trained_model.set_train(False)
        probs = trained_model(input_tensor).asnumpy()[0]

        pred_idx_0based = int(np.argmax(probs))
        pred_conf = float(probs[pred_idx_0based])

        idx_to_skill = {idx - 1: skill for skill, idx in skill_to_idx.items()}
        true_skill_label = idx_to_skill.get(int(target_skill), f'idx_{int(target_skill)}')
        pred_skill_label = idx_to_skill.get(pred_idx_0based, f'idx_{pred_idx_0based}')

        print('\nSingle-instance inference result')
        print(f'True next-skill index (0-based): {int(target_skill)} -> {true_skill_label}')
        print(f'Predicted next-skill index (0-based): {pred_idx_0based} -> {pred_skill_label}')
        print(f'Confidence: {pred_conf:.4f}')

        top_k = 5
        top_indices = np.argsort(probs)[-top_k:][::-1]
        print('\nTop-5 predicted skills:')
        for rank, idx0 in enumerate(top_indices, 1):
            skill_label = idx_to_skill.get(int(idx0), f'idx_{int(idx0)}')
            print(f'  {rank}. {skill_label} (idx={int(idx0)}, p={float(probs[idx0]):.4f})')

        print(f'Match: {pred_idx_0based == int(target_skill)}')



Single-instance inference result
True next-skill index (0-based): 64 -> 9;15
Predicted next-skill index (0-based): 1 -> 47
Confidence: 0.5041

Top-5 predicted skills:
  1. 47 (idx=1, p=0.5041)
  2. 16;17 (idx=21, p=0.5030)
  3. 41542 (idx=111, p=0.5024)
  4. 146812 (idx=467, p=0.5024)
  5. 18 (idx=15, p=0.5023)
Match: False


In [6]:
# Post-training evaluation and performance benchmark
import time

if 'trained_model' not in globals() or trained_model is None:
    print('Run the partial training cell first to create trained_model.')
elif 'test_sequences' not in globals() or len(test_sequences) == 0:
    print('test_sequences not found. Re-run data split in the partial training cell.')
else:
    eval_data = prepare_sequences_for_training(test_sequences, max_length=50)

    if len(eval_data) == 0:
        print('No valid evaluation samples found.')
    else:
        max_eval_samples = 10000  # Increase if you want stricter evaluation
        batch_size = 128
        eval_subset = eval_data[:max_eval_samples]

        trained_model.set_train(False)

        top1_correct = 0
        top5_correct = 0
        confidences = []
        total = len(eval_subset)

        start_time = time.time()

        for i in range(0, total, batch_size):
            batch = eval_subset[i:i + batch_size]
            batch_x = np.stack([np.clip(x, 0, num_skills * 2) for x, _ in batch]).astype(np.int32)
            batch_y = np.array([y for _, y in batch], dtype=np.int32)

            probs = trained_model(Tensor(batch_x, ms.int32)).asnumpy()

            pred_top1 = np.argmax(probs, axis=1)
            top1_correct += int(np.sum(pred_top1 == batch_y))

            pred_top5 = np.argsort(probs, axis=1)[:, -5:]
            top5_correct += int(np.sum([target in row for target, row in zip(batch_y, pred_top5)]))

            confidences.extend(np.max(probs, axis=1).tolist())

        elapsed = time.time() - start_time

        top1_acc = top1_correct / total
        top5_acc = top5_correct / total
        avg_conf = float(np.mean(confidences)) if confidences else 0.0
        throughput = total / elapsed if elapsed > 0 else 0.0
        latency_ms = (elapsed / total) * 1000 if total > 0 else 0.0

        print('\nEvaluation + Performance Summary')
        print(f'Samples evaluated: {total}')
        print(f'Top-1 Accuracy: {top1_acc:.4f}')
        print(f'Top-5 Accuracy: {top5_acc:.4f}')
        print(f'Average confidence (max prob): {avg_conf:.4f}')
        print(f'Total inference time: {elapsed:.2f} s')
        print(f'Average latency: {latency_ms:.3f} ms/sample')
        print(f'Throughput: {throughput:.2f} samples/s')



Evaluation + Performance Summary
Samples evaluated: 10000
Top-1 Accuracy: 0.0380
Top-5 Accuracy: 0.0682
Average confidence (max prob): 0.5041
Total inference time: 0.84 s
Average latency: 0.084 ms/sample
Throughput: 11867.01 samples/s


In [7]:
# File: dkt_full_training_fixed.py
import numpy as np
import pandas as pd
import mindspore as ms
import mindspore.nn as nn
import mindspore.ops as ops
from mindspore import Tensor
from mindspore.dataset import GeneratorDataset
import pickle
import time
import os
from pathlib import Path
from datetime import datetime
import json

# Set context
ms.set_context(device_target="GPU")

def resolve_work_dir():
    candidates = [Path.cwd(), Path.cwd() / "dkt"]
    for candidate in candidates:
        if (candidate / "assistments_2009_2010.csv").exists():
            return candidate
    return Path.cwd()

WORK_DIR = resolve_work_dir()

def load_and_preprocess_data_full(file_path):
    """
    Load ALL data for full training
    """
    print("Loading FULL dataset...")
    print(f"Start time: {datetime.now().strftime('%H:%M:%S')}")

    df = pd.read_csv(file_path)
    print(f"Dataset shape: {df.shape}")

    # Use all skills
    skill_column = 'list_skill_ids'
    df[skill_column] = df[skill_column].fillna(df['problem_id'].astype(str))
    df[skill_column] = df[skill_column].astype(str)
    df['skill'] = df[skill_column].apply(lambda x: str(x).split(',')[0] if ',' in str(x) else str(x))

    # Convert correct to binary
    df['correct'] = df['correct'].fillna(0).astype(int)

    # Filter out invalid skills
    df = df[df['skill'] != 'nan']
    df = df[df['skill'] != '']

    # Create skill mapping for ALL skills
    print("\nCreating skill mapping for ALL skills...")
    all_skills = df['skill'].unique().tolist()
    skill_to_idx = {skill: idx + 1 for idx, skill in enumerate(all_skills)}
    num_skills = len(all_skills)

    print(f"✓ Using ALL {num_skills:,} skills")
    print(f"Number of unique students: {df['user_id'].nunique():,}")
    print(f"Total interactions: {len(df):,}")

    # Save skill mapping for later use
    with open(WORK_DIR / 'skill_mapping_full.pkl', 'wb') as f:
        pickle.dump(skill_to_idx, f)
    print(f"✓ Saved full skill mapping to '{WORK_DIR / 'skill_mapping_full.pkl'}'")

    # Sort and prepare sequences
    df = df.sort_values(['user_id', 'order_id'])

    # Process in chunks to save memory
    chunk_size = 1000
    student_sequences = []

    print("\nProcessing student sequences...")
    student_groups = list(df.groupby('user_id'))

    for i, (user_id, group) in enumerate(student_groups):
        skills = []
        corrects = []

        for _, row in group.iterrows():
            skill = row['skill']
            correct = row['correct']

            if skill in skill_to_idx:
                skills.append(skill_to_idx[skill])
                corrects.append(correct)

        if len(skills) >= 2:  # Need at least 2 interactions
            # Create interaction encoding
            interactions = []
            for skill_idx, correct in zip(skills, corrects):
                skill_idx_0based = skill_idx - 1
                token = skill_idx_0based * 2 + correct + 1
                interactions.append(token)

            student_sequences.append(interactions)

        if i % 1000 == 0 and i > 0:
            print(f"  Processed {i:,}/{len(student_groups):,} students")

    print(f"\n✓ Created {len(student_sequences):,} student sequences")
    print(f"Average sequence length: {np.mean([len(s) for s in student_sequences]):.2f}")

    return student_sequences, num_skills, skill_to_idx

class MemoryEfficientDKTModel(nn.Cell):
    """Memory efficient DKT model for full training"""
    def __init__(self, num_skills, hidden_size=128, embedding_dim=100):
        super(MemoryEfficientDKTModel, self).__init__()

        self.vocab_size = num_skills * 2 + 1
        self.num_skills = num_skills

        # Use embedding with padding
        self.embedding = nn.Embedding(self.vocab_size, embedding_dim, padding_idx=0)

        # GRU
        self.gru = nn.GRU(
            embedding_dim,
            hidden_size,
            num_layers=1,
            batch_first=True,
            bidirectional=False
        )

        # Output layer
        self.fc = nn.Dense(hidden_size, num_skills)
        self.sigmoid = nn.Sigmoid()

        print(f"\nModel Configuration for Full Training:")
        print(f"  Number of skills: {num_skills:,}")
        print(f"  Vocabulary size: {self.vocab_size:,}")
        print(f"  Embedding dim: {embedding_dim}")
        print(f"  Hidden size: {hidden_size}")

    def construct(self, x):
        embedded = self.embedding(x)
        gru_output, _ = self.gru(embedded)
        last_output = gru_output[:, -1, :]
        output = self.fc(last_output)
        probabilities = self.sigmoid(output)
        return probabilities

def save_training_summary(num_skills, train_sequences, test_sequences, save_dir="checkpoints_full"):
    """Save training summary"""
    # Create directory if it doesn't exist
    os.makedirs(save_dir, exist_ok=True)

    summary = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "num_skills": num_skills,
        "vocab_size": num_skills * 2 + 1,
        "train_sequences": len(train_sequences),
        "test_sequences": len(test_sequences),
        "avg_sequence_length": float(np.mean([len(s) for s in train_sequences])),
        "total_interactions": sum([len(s) for s in train_sequences]),
        "model_config": {
            "hidden_size": 128,
            "embedding_dim": 100,
            "max_sequence_length": 100
        }
    }

    summary_path = f"{save_dir}/training_summary.json"
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)

    print(f"\nTraining summary saved to: {summary_path}")
    print("Summary:")
    for key, value in summary.items():
        if key != "model_config":
            print(f"  {key}: {value:,}" if isinstance(value, int) else f"  {key}: {value}")

def train_full_model_simple(train_sequences, num_skills, epochs=5,
                           batch_size=32, learning_rate=0.001,
                           save_dir="checkpoints_full"):
    """Simplified full training for overnight run"""

    # Create save directory
    os.makedirs(save_dir, exist_ok=True)

    print("\nPreparing training data...")
    max_length = 100

    # Process sequences efficiently
    train_data = []
    for seq in train_sequences:
        if len(seq) < 2:
            continue

        # Create windows
        for start in range(0, min(len(seq) - 1, max_length * 2)):
            end = min(start + max_length, len(seq))
            window = seq[start:end]

            if len(window) < 2:
                continue

            input_seq = window[:-1]
            last_token = window[-1]
            target_skill = (last_token - 1) // 2

            if target_skill < 0 or target_skill >= num_skills:
                continue

            padded_input = np.zeros(max_length - 1, dtype=np.int32)
            if len(input_seq) > 0:
                padded_input[-len(input_seq):] = input_seq

            train_data.append((padded_input, target_skill))

    print(f"✓ Training samples: {len(train_data):,}")

    # Limit to 200K samples for faster training
    if len(train_data) > 200000:
        print(f"Using first 200,000 samples for manageable training")
        train_data = train_data[:200000]

    # Create dataset
    dataset = GeneratorDataset(
        source=train_data,
        column_names=['input', 'target'],
        shuffle=True
    ).batch(batch_size, drop_remainder=True)

    # Initialize model
    print("\nInitializing model...")
    model = MemoryEfficientDKTModel(num_skills, hidden_size=128, embedding_dim=100)

    # Loss function
    class DKTFullLoss(nn.Cell):
        def __init__(self, model, num_skills):
            super(DKTFullLoss, self).__init__()
            self.model = model
            self.num_skills = num_skills
            self.loss_fn = nn.BCELoss(reduction='mean')
            self.one_hot = ops.OneHot()

        def construct(self, x, target_skill):
            predictions = self.model(x)
            target_one_hot = self.one_hot(target_skill, self.num_skills,
                                          Tensor(1.0, ms.float32), Tensor(0.0, ms.float32))
            loss = self.loss_fn(predictions, target_one_hot)
            return loss

    loss_net = DKTFullLoss(model, num_skills)
    optimizer = nn.Adam(model.trainable_params(), learning_rate=learning_rate)
    train_network = nn.TrainOneStepCell(loss_net, optimizer)
    train_network.set_train()

    def train_step(x, target):
        return train_network(x, target)

    # Training loop
    print(f"\nStarting training for {epochs} epochs...")
    print(f"Batch size: {batch_size}")
    print(f"Learning rate: {learning_rate}")
    print("-" * 60)

    start_time = time.time()

    for epoch in range(epochs):
        epoch_loss = 0
        step_count = 0

        for batch in dataset.create_tuple_iterator():
            inputs, targets = batch
            inputs = Tensor(inputs, ms.int32)
            targets = Tensor(targets, ms.int32)

            # Clip inputs
            max_token = num_skills * 2
            inputs = ops.clip_by_value(inputs, Tensor(0, ms.int32), Tensor(max_token, ms.int32))

            try:
                loss = train_step(inputs, targets)
                epoch_loss += loss.asnumpy()
                step_count += 1

                if step_count % 100 == 0:
                    elapsed = (time.time() - start_time) / 60
                    print(f"[E{epoch+1}] Step {step_count}, Loss: {loss.asnumpy():.4f}, Time: {elapsed:.1f}min")

            except Exception as e:
                print(f"Error at step {step_count}: {e}")
                continue

        if step_count > 0:
            avg_loss = epoch_loss / step_count
            elapsed = (time.time() - start_time) / 60

            print(f"\nEpoch {epoch+1}/{epochs} complete!")
            print(f"Average Loss: {avg_loss:.4f}")
            print(f"Time: {elapsed:.1f} minutes")

            # Save checkpoint
            checkpoint_path = f"{save_dir}/epoch_{epoch+1}.ckpt"
            ms.save_checkpoint(model, checkpoint_path)
            print(f"✓ Checkpoint saved: {checkpoint_path}")

    # Final save
    final_checkpoint = f"{save_dir}/dkt_model_final.ckpt"
    ms.save_checkpoint(model, final_checkpoint)

    # Export to MindIR
    dummy_input = Tensor(np.random.randint(0, num_skills * 2 + 1, size=(1, max_length-1)), ms.int32)
    final_mindir = WORK_DIR / "dkt_model_full"
    ms.export(model, dummy_input, file_name=str(final_mindir), file_format="MINDIR")

    print(f"\n{'='*60}")
    print("TRAINING COMPLETE!")
    print(f"{'='*60}")
    print(f"Final checkpoint: {final_checkpoint}")
    print(f"Final MindIR model: {final_mindir}.mindir")
    print(f"Total training time: {(time.time() - start_time)/3600:.2f} hours")

    return model

# Main execution
if __name__ == "__main__":
    print("=" * 70)
    print("DKT FULL TRAINING - SIMPLIFIED FOR OVERNIGHT")
    print("=" * 70)

    try:
        # Load data
        file_path = WORK_DIR / "assistments_2009_2010.csv"
        print("\nLoading data...")
        sequences, num_skills, skill_to_idx = load_and_preprocess_data_full(file_path)

        # Split data
        np.random.seed(42)
        indices = np.arange(len(sequences))
        np.random.shuffle(indices)
        split_idx = int(len(sequences) * 0.8)
        train_indices = indices[:split_idx]
        test_indices = indices[split_idx:]

        train_sequences = [sequences[i] for i in train_indices]
        test_sequences = [sequences[i] for i in test_indices]

        print(f"\nData Split:")
        print(f"  Training sequences: {len(train_sequences):,}")
        print(f"  Testing sequences: {len(test_sequences):,}")

        # Save test sequences
        with open(WORK_DIR / 'test_sequences_full.pkl', 'wb') as f:
            pickle.dump(test_sequences, f)
        print(f"✓ Test sequences saved to {WORK_DIR / 'test_sequences_full.pkl'}")

        # Save training summary
        save_training_summary(num_skills, train_sequences, test_sequences, save_dir=str(WORK_DIR / "checkpoints_full"))

        # Start training
        print(f"\n{'='*70}")
        print("READY TO START TRAINING")
        print(f"{'='*70}")

        # Ask for confirmation
        response = input("Start overnight training? (yes/no): ")
        if response.lower() != 'yes':
            print("Training cancelled")
            exit()

        print("\nStarting training in 5 seconds...")
        time.sleep(5)

        # Train
        trained_model = train_full_model_simple(
            train_sequences,
            num_skills,
            epochs=10,  # 10 epochs for full training
            batch_size=64,
            learning_rate=0.001,
            save_dir=str(WORK_DIR / "checkpoints_full")
        )

        print("\n" + "="*70)
        print("OVERNIGHT TRAINING SETUP COMPLETE!")
        print("="*70)
        print("The model will train overnight with these settings:")
        print("- 22,779 skills")
        print("- 200,000 training samples")
        print("- 10 epochs")
        print("- Checkpoints saved every epoch")
        print(f"- Start time: {datetime.now().strftime('%H:%M:%S')}")

    except KeyboardInterrupt:
        print("\n\nTraining interrupted by user")

    except Exception as e:
        print(f"\n❌ Error: {e}")
        import traceback
        traceback.print_exc()

DKT FULL TRAINING - SIMPLIFIED FOR OVERNIGHT

Loading data...
Loading FULL dataset...
Start time: 10:53:08
Dataset shape: (1011079, 20)

Creating skill mapping for ALL skills...
✓ Using ALL 22,779 skills
Number of unique students: 8,519
Total interactions: 1,011,079
✓ Saved full skill mapping to '/workspace/workspace/dkt/skill_mapping_full.pkl'

Processing student sequences...
  Processed 1,000/8,519 students
  Processed 2,000/8,519 students
  Processed 3,000/8,519 students
  Processed 4,000/8,519 students
  Processed 5,000/8,519 students
  Processed 6,000/8,519 students
  Processed 7,000/8,519 students
  Processed 8,000/8,519 students

✓ Created 8,344 student sequences
Average sequence length: 121.15

Data Split:
  Training sequences: 6,675
  Testing sequences: 1,669
✓ Test sequences saved to /workspace/workspace/dkt/test_sequences_full.pkl

Training summary saved to: /workspace/workspace/dkt/checkpoints_full/training_summary.json
Summary:
  timestamp: 2026-02-13 10:54:00
  num_skills

Start overnight training? (yes/no):  yes



Starting training in 5 seconds...

Preparing training data...
✓ Training samples: 467,034
Using first 200,000 samples for manageable training

Initializing model...

Model Configuration for Full Training:
  Number of skills: 22,779
  Vocabulary size: 45,559
  Embedding dim: 100
  Hidden size: 128

Starting training for 10 epochs...
Batch size: 64
Learning rate: 0.001
------------------------------------------------------------


[ERROR] CORE(381,7605dd677680,python3.7):2026-02-13-10:54:14.140.630 [mindspore/core/utils/file_utils.cc:253] GetRealPath] Get realpath failed, path[/tmp/ipykernel_381/2817906689.py]
[ERROR] CORE(381,7605dd677680,python3.7):2026-02-13-10:54:14.141.258 [mindspore/core/utils/file_utils.cc:253] GetRealPath] Get realpath failed, path[/tmp/ipykernel_381/2817906689.py]


[E1] Step 100, Loss: 0.0035, Time: 0.0min
[E1] Step 200, Loss: 0.0017, Time: 0.0min
[E1] Step 300, Loss: 0.0011, Time: 0.0min
[E1] Step 400, Loss: 0.0009, Time: 0.1min
[E1] Step 500, Loss: 0.0007, Time: 0.1min
[E1] Step 600, Loss: 0.0006, Time: 0.1min
[E1] Step 700, Loss: 0.0006, Time: 0.1min
[E1] Step 800, Loss: 0.0005, Time: 0.1min
[E1] Step 900, Loss: 0.0005, Time: 0.1min
[E1] Step 1000, Loss: 0.0005, Time: 0.1min
[E1] Step 1100, Loss: 0.0004, Time: 0.1min
[E1] Step 1200, Loss: 0.0004, Time: 0.1min
[E1] Step 1300, Loss: 0.0004, Time: 0.2min
[E1] Step 1400, Loss: 0.0004, Time: 0.2min
[E1] Step 1500, Loss: 0.0004, Time: 0.2min
[E1] Step 1600, Loss: 0.0004, Time: 0.2min
[E1] Step 1700, Loss: 0.0004, Time: 0.2min
[E1] Step 1800, Loss: 0.0004, Time: 0.2min
[E1] Step 1900, Loss: 0.0004, Time: 0.2min
[E1] Step 2000, Loss: 0.0004, Time: 0.2min
[E1] Step 2100, Loss: 0.0004, Time: 0.3min
[E1] Step 2200, Loss: 0.0004, Time: 0.3min
[E1] Step 2300, Loss: 0.0004, Time: 0.3min
[E1] Step 2400, Loss

To Run This Overnight:

# 1. Run the training script (will take 6-12 hours)
nohup python dkt_full_training_fixed.py > full_training.log 2>&1 &

# 2. Monitor progress
tail -f full_training.log

# 3. Check if it's still running
ps aux | grep dkt_full_training

# 4. Check memory usage (optional)
watch -n 60 'free -h'

If Training Gets Interrupted:
# Quick resume script
import glob
import mindspore as ms

# Find latest checkpoint
checkpoints = glob.glob("checkpoints_full/*.ckpt")
if checkpoints:
    latest = sorted(checkpoints)[-1]
    print(f"Resume from: {latest}")
    # Load and continue training